### TODO: Miles to km (?)

### TODO: Provide a detailed description of the trip dataset such that there are no pending questions.

### TODO: Evaluate Aggregation Logic especially in regards to Spatial Analysis and Prediction tasks

### TODO: Make Markdown Text look good

### TODO: Add column description from website here as Markdown

### TODO: Add Outlier Analysis

In [1]:
import pandas as pd
import numpy as np
import h3

## What happened before uploading the CSV:
- Filtering the data for:
    - Pickup/Dropoff Census Tract is not null
    - Trip Seconds/Miles is not 0
    - Removing unneccasssary columns: Fare, Tips, Tolls, Extras, Payment Type, Pickup/Dropoff Centroid Location
- resulting data with 16 columns and 6.041.177 rows

In [3]:
taxi_data = pd.read_csv("../data/Taxi_Trips_big.csv")
taxi_data

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,...,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
0,7bb35d509b4a23a4d5deae7018a80bdd5f87c952,8b07f9156e568a37d362463c84dbd1118b4eeb753bae50...,06/01/2026 12:00:00 AM,06/01/2026 12:00:00 AM,300.000,"1,2",NaN,NaN,32.0,32.0,...,"$1,00","$11,25",Credit Card,Choice Taxi Association Inc,"41,878865584","-87,625192142",POINT (-87.6251921424 41.8788655841),"41,878865584","-87,625192142",POINT (-87.6251921424 41.8788655841)
1,6630e8872591a9a915ad804f5ce3b34a4654663a,6cf97b9e8e3e1c183c090cdd868df381de772a52987239...,06/01/2026 12:00:00 AM,06/01/2026 12:00:00 AM,445.000,"1,5",NaN,NaN,32.0,28.0,...,"$0,00","$8,92",Mobile,Flash Cab,"41,878865584","-87,625192142",POINT (-87.6251921424 41.8788655841),"41,874005383","-87,66351755",POINT (-87.6635175498 41.874005383)
2,51e2af26f6d9c93afc6e4f1b26a9bf6a3fa7700d,3665a72ee495b03f4dae72307dc6e5e58e21518f77d8e6...,06/01/2026 12:00:00 AM,06/01/2026 12:00:00 AM,0.000,0,NaN,NaN,28.0,28.0,...,"$0,00","$10,38",Credit Card,Transit Administrative Center Inc,"41,874005383","-87,66351755",POINT (-87.6635175498 41.874005383),"41,874005383","-87,66351755",POINT (-87.6635175498 41.874005383)
3,fe861de7a4b804177aa770afd5134dd22758a593,780424fddffb94d4d8541780c7ad1a953ef87d12a20def...,06/01/2026 12:00:00 AM,06/01/2026 12:00:00 AM,4.000,0,NaN,NaN,72.0,72.0,...,"$0,00","$48,60",Credit Card,Globe Taxi,"41,713148612","-87,675075312",POINT (-87.6750753124 41.713148612),"41,713148612","-87,675075312",POINT (-87.6750753124 41.713148612)
4,e2627bcf1c079e0b086cb8dbfc34d6ea955a899a,d5c821e78bf25431862f40219794e4d023679198afeb74...,06/01/2026 12:00:00 AM,06/01/2026 12:00:00 AM,380.000,"1,3",NaN,NaN,32.0,8.0,...,"$0,00","$8,33",Mobile,Taxicab Insurance Agency Llc,"41,878865584","-87,625192142",POINT (-87.6251921424 41.8788655841),"41,899602111","-87,633308037",POINT (-87.6333080367 41.899602111)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16086250,fc08277cee8647cb9a4eb700da02ce287029ca2a,3618045f9110d4d88482266ade23659c1a50d32ac37f20...,01/01/2024 12:00:00 AM,01/01/2024 12:15:00 AM,840.000,"7,1",NaN,NaN,41.0,32.0,...,"$0,00","$20,25",Cash,Taxi Affiliation Services,"41,794090253","-87,592310855",POINT (-87.592310855 41.794090253),"41,878865584","-87,625192142",POINT (-87.6251921424 41.8788655841)
16086251,be2ab4c001613a36d473357c2f81c214554ef310,73052f4ccaf4e0fa9178722e491f8e5eda869f56e08aa4...,01/01/2024 12:00:00 AM,01/01/2024 12:45:00 AM,2.820,0,NaN,NaN,28.0,NaN,...,"$0,00","$60,50",Unknown,Taxi Affiliation Services,"41,874005383","-87,66351755",POINT (-87.6635175498 41.874005383),NaN,NaN,NaN
16086252,bdc420394ce5e864465df0a361dfbe95a4e228c4,389f01c14b097ed951468ff163ccc71ebcb99a27e523e9...,01/01/2024 12:00:00 AM,01/01/2024 12:30:00 AM,1.369,"3,07",NaN,NaN,8.0,24.0,...,"$1,00","$26,00",Mobile,Medallion Leasin,"41,899602111","-87,633308037",POINT (-87.6333080367 41.899602111),"41,901206994","-87,676355989",POINT (-87.6763559892 41.9012069941)
16086253,70679bacef2403edfa376cd42f2b04037e7534c1,8123ec5a70681a139fcd93fd477eec0ee05dda82987748...,01/01/2024 12:00:00 AM,01/01/2024 12:15:00 AM,584.000,"0,43",NaN,NaN,8.0,8.0,...,"$1,50","$7,75",Cash,Globe Taxi,"41,899602111","-87,633308037",POINT (-87.6333080367 41.899602111),"41,899602111","-87,633308037",POINT (-87.6333080367 41.899602111)


In [5]:
taxi_data.info()
#taxi_data.describe()

<class 'pandas.DataFrame'>
RangeIndex: 16086255 entries, 0 to 16086254
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     str    
 1   Taxi ID                     str    
 2   Trip Start Timestamp        str    
 3   Trip End Timestamp          str    
 4   Trip Seconds                float64
 5   Trip Miles                  str    
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Fare                        str    
 11  Tips                        str    
 12  Tolls                       str    
 13  Extras                      str    
 14  Trip Total                  str    
 15  Payment Type                str    
 16  Company                     str    
 17  Pickup Centroid Latitude    str    
 18  Pickup Centroid Longitude   str    
 19  Pickup Centroid Location    st

## 1. Delete unneccessary columns
As the features *Fare*, *Tips*, *Tolls*, *Extras* and *Payment Type* are not used for analysis, they are deleted from the dataset.
The features *Pickup Centroid Location* and *Dropoff Centroid Location* are redundant and therefore deleted as well.

In [8]:
cols_to_drop = ["Fare", "Tips", "Tolls", "Extras", "Payment Type", "Pickup Centroid Location", "Dropoff Centroid  Location"]

taxi_data = taxi_data.drop(columns=cols_to_drop)
taxi_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 16086255 entries, 0 to 16086254
Data columns (total 16 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     str    
 1   Taxi ID                     str    
 2   Trip Start Timestamp        str    
 3   Trip End Timestamp          str    
 4   Trip Seconds                float64
 5   Trip Miles                  str    
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Trip Total                  str    
 11  Company                     str    
 12  Pickup Centroid Latitude    str    
 13  Pickup Centroid Longitude   str    
 14  Dropoff Centroid Latitude   str    
 15  Dropoff Centroid Longitude  str    
dtypes: float64(5), str(11)
memory usage: 6.2 GB


## 2. Change Data types from string to numeric

In [9]:
# Columns to fix data type
cols_to_fix = [
    'Pickup Centroid Longitude', 'Pickup Centroid Latitude',
    'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude',
    'Trip Total', 'Trip Miles'
]

for col in cols_to_fix:
    
    if col == 'Trip Total':
        # Remove the dollar sign and commas from the Trip Total column
        taxi_data[col] = taxi_data[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
    elif col == 'Trip Miles' or col in ['Pickup Centroid Longitude', 'Pickup Centroid Latitude', 'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude']:
        # Replace the comma with a standard decimal point
        taxi_data[col] = taxi_data[col].astype(str).str.replace(',', '.', regex=False)

    # Convert to numeric
    taxi_data[col] = pd.to_numeric(taxi_data[col], errors='coerce')

In [13]:
taxi_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 16086255 entries, 0 to 16086254
Data columns (total 16 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     str    
 1   Taxi ID                     str    
 2   Trip Start Timestamp        str    
 3   Trip End Timestamp          str    
 4   Trip Seconds                float64
 5   Trip Miles                  float64
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Trip Total                  float64
 11  Company                     str    
 12  Pickup Centroid Latitude    float64
 13  Pickup Centroid Longitude   float64
 14  Dropoff Centroid Latitude   float64
 15  Dropoff Centroid Longitude  float64
dtypes: float64(11), str(5)
memory usage: 5.4 GB


# 2. Null Value Analysis

In [10]:
# Find Null Values and add percentage of NaN values for each column
is_na_df = taxi_data.isna().sum()
is_na_df = pd.DataFrame(is_na_df, columns=['NaN Count'])
is_na_df['Total Count'] = len(taxi_data)
is_na_df['NaN Percentage'] = (is_na_df['NaN Count'] / is_na_df['Total Count']) * 100

# Check for any NaN values in all columns
print("NaN values before conversion:")
print(is_na_df)

NaN values before conversion:
                            NaN Count  Total Count  NaN Percentage
Trip ID                             0     16086255        0.000000
Taxi ID                            12     16086255        0.000075
Trip Start Timestamp                0     16086255        0.000000
Trip End Timestamp                427     16086255        0.002654
Trip Seconds                     3081     16086255        0.019153
Trip Miles                        157     16086255        0.000976
Pickup Census Tract           8891026     16086255       55.270950
Dropoff Census Tract          9102541     16086255       56.585831
Pickup Community Area          445622     16086255        2.770204
Dropoff Community Area        1414559     16086255        8.793588
Trip Total                      32557     16086255        0.202390
Company                             0     16086255        0.000000
Pickup Centroid Latitude       437472     16086255        2.719539
Pickup Centroid Longitude      4

As the columns *Taxi_ID*, *Trip End Timestamp*, *Trip Seconds*, *Trip Miles* and *Trip Total* are needed for the analysis, and the number of null values are relatively small, those datapoints are simply deleted from the dataset.

In [ ]:
taxi_data_v2 = taxi_data.dropna(subset=['Taxi ID', 'Trip End Timestamp', 
                                                    'Trip Seconds', 'Trip Miles',
                                                    'Trip Total'])
#taxi_data_v2.info()

<class 'pandas.DataFrame'>
Index: 16051540 entries, 0 to 16086254
Data columns (total 16 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     str    
 1   Taxi ID                     str    
 2   Trip Start Timestamp        str    
 3   Trip End Timestamp          str    
 4   Trip Seconds                float64
 5   Trip Miles                  float64
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Trip Total                  float64
 11  Company                     str    
 12  Pickup Centroid Latitude    float64
 13  Pickup Centroid Longitude   float64
 14  Dropoff Centroid Latitude   float64
 15  Dropoff Centroid Longitude  float64
dtypes: float64(11), str(5)
memory usage: 5.5 GB


For the spatial entities the number of null values is relatively big, especially for *Census Tract*, so further analysis is neccessary.

In [14]:
spatial_cols = [
    "Pickup Census Tract",
    "Dropoff Census Tract",
    "Pickup Community Area",
    "Dropoff Community Area",
    "Pickup Centroid Latitude",
    "Pickup Centroid Longitude",
    "Dropoff Centroid Latitude",
    "Dropoff Centroid Longitude",
]
null_mask = taxi_data_v2[spatial_cols].isna()

conditional_overlap = pd.DataFrame(
    index=spatial_cols,
    columns=spatial_cols,
    dtype=float
)

for c1 in spatial_cols:
    missing_c1 = null_mask[c1].sum()

    for c2 in spatial_cols:
        if missing_c1 == 0:
            conditional_overlap.loc[c1, c2] = 0
        else:
            conditional_overlap.loc[c1, c2] = (
                (null_mask[c1] & null_mask[c2]).sum() / missing_c1
            )

conditional_overlap = conditional_overlap.round(3)

conditional_overlap

,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,Pickup Centroid Latitude,Pickup Centroid Longitude,Dropoff Centroid Latitude,Dropoff Centroid Longitude
Pickup Census Tract,1.000,0.997,0.049,0.121,0.049,0.049,0.121,0.121
Dropoff Census Tract,0.974,1.000,0.045,0.144,0.045,0.045,0.144,0.144
Pickup Community Area,0.977,0.911,1.000,0.671,0.982,0.982,0.655,0.655
Dropoff Community Area,0.759,0.929,0.211,1.000,0.207,0.207,0.941,0.941
Pickup Centroid Latitude,0.995,0.927,1.000,0.668,1.000,1.000,0.667,0.667
Pickup Centroid Longitude,0.995,0.927,1.000,0.668,1.000,1.000,0.667,0.667
Dropoff Centroid Latitude,0.806,0.987,0.220,1.000,0.219,0.219,1.000,1.000
Dropoff Centroid Longitude,0.806,0.987,0.220,1.000,0.219,0.219,1.000,1.000


The overlap-matrix shows, that the data points which have null values at *Pickup/ Dropoff Community Area* mostly have null values at the corresponding *Census Tract* and *Longitude/ Latitude* values. Thefore we cannot use those datapoints and delete them.

The data points with null values at *Census Tract* make up for more than half of the dataset. That is because for privacy, the Census Tract is not shown for some trips and/ or is blank for locations outside Chicago. 

As the *Community Area* and *Longitude/Latitude* is given for most of those data points, we will split tha dataset at the end of preparation. For Census Tract Analysis we will build one dataset which has no Null Values at Census Tract and for *Community Area* Analysis we will work with the much bigger dataset where Census Tract can be null.

In [18]:
taxi_data_v3 = taxi_data_v2.dropna(subset=['Pickup Community Area', 'Dropoff Community Area'])

In [19]:
taxi_data_v3.isna().sum()

Trip ID                             0
Taxi ID                             0
Trip Start Timestamp                0
Trip End Timestamp                  0
Trip Seconds                        0
Trip Miles                          0
Pickup Census Tract           7659353
Dropoff Census Tract          7659353
Pickup Community Area               0
Dropoff Community Area              0
Trip Total                          0
Company                             0
Pickup Centroid Latitude            0
Pickup Centroid Longitude           0
Dropoff Centroid Latitude           0
Dropoff Centroid Longitude          0
dtype: int64

## 3. Add Column with H3 index

In [23]:
taxi_data_v3['h3_index_pickup_7'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 7) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Pickup Centroid Latitude'], 
        taxi_data_v3['Pickup Centroid Longitude']
    )
]
taxi_data_v3['h3_index_dropoff_7'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 7) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Dropoff Centroid Latitude'], 
        taxi_data_v3['Dropoff Centroid Longitude']
    )
]

taxi_data_v3['h3_index_pickup_8'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 8) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Pickup Centroid Latitude'], 
        taxi_data_v3['Pickup Centroid Longitude']
    )
]
taxi_data_v3['h3_index_dropoff_8'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 8) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Dropoff Centroid Latitude'], 
        taxi_data_v3['Dropoff Centroid Longitude']
    )
]

taxi_data_v3['h3_index_pickup_9'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 9) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Pickup Centroid Latitude'], 
        taxi_data_v3['Pickup Centroid Longitude']
    )
]
taxi_data_v3['h3_index_dropoff_9'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 9) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Dropoff Centroid Latitude'], 
        taxi_data_v3['Dropoff Centroid Longitude']
    )
]

In [ ]:
# Check for any NaN values in the new H3 index columns
# TODO: Add percentage of NaN values in the H3 index columns
print(taxi_data_v3[['h3_index_pickup_7', 
                 'h3_index_dropoff_7',
                 'h3_index_pickup_8',
                 'h3_index_dropoff_8',
                 'h3_index_pickup_9',
                 'h3_index_dropoff_9',
                 'Pickup Census Tract', 
                 'Dropoff Census Tract']].isna().sum())

columns_to_drop = [
    'Pickup Centroid Longitude', 'Pickup Centroid Latitude',
    'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude'
]

#TODO: ask if anybody still needs Long/lat values else drop them 

h3_index_pickup_7             0
h3_index_dropoff_7            0
h3_index_pickup_8             0
h3_index_dropoff_8            0
h3_index_pickup_9             0
h3_index_dropoff_9            0
Pickup Census Tract     7659353
Dropoff Census Tract    7659353
dtype: int64


## Save the dataset as parquet
The big dataset is being used for Commuinty Area Analysis.

In [28]:
taxi_data_v3.to_parquet(
    "../data/processed/taxi_data_processed_big.parquet"
)

The small dataset is being used for Census Tract Analysis.

In [27]:
taxi_data_small = taxi_data_v3.dropna(subset=['Pickup Census Tract', 'Dropoff Census Tract'])

In [29]:
taxi_data_small.to_parquet(
    "../data/processed/taxi_data_processed_small.parquet"
)